In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Load the datasets (Adjust paths if running locally)
train_df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

# Save PassengerId for the final submission file
passenger_ids = test_df['PassengerId']

# Combine datasets temporarily to ensure consistent preprocessing and feature engineering
df = pd.concat([train_df.drop('Survived', axis=1), test_df], axis=0).reset_index(drop=True)

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

Training data shape: (891, 12)
Test data shape: (418, 11)


In [2]:
# 1. Extract Titles (only if 'Name' column is present in df)
if 'Name' in df.columns:
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
    df['Title'] = df['Title'].replace(rare_titles, 'Rare')
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# 2. Family Size and Solitude Features (only if columns exist)
if 'SibSp' in df.columns and 'Parch' in df.columns:
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 3. Impute Missing Values
if 'Age' in df.columns:
    df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

if 'Embarked' in df.columns:
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

if 'Fare' in df.columns:
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# 4. Drop columns only if they still exist in df
columns_to_drop = [col for col in ['PassengerId', 'Name', 'Ticket', 'Cabin'] if col in df.columns]
if columns_to_drop:
    df = df.drop(columns_to_drop, axis=1)

print("Missing values remaining:\n", df.isnull().sum())

Missing values remaining:
 Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked      0
Title         0
FamilySize    0
IsAlone       0
dtype: int64


In [3]:
# One-hot encode categoricals and drop the first category to avoid multicollinearity
df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title'], drop_first=True)

# Split back into training and test sets
X_train = df.iloc[:len(train_df)].copy()
X_test = df.iloc[len(train_df):].copy()
y_train = train_df['Survived']

print(f"Processed training features shape: {X_train.shape}")

Processed training features shape: (891, 14)


In [4]:
scaler = StandardScaler()

# Fit the scaler on training data and transform both training and test features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
# Define hyperparameter grid to explore
param_grid = {
    'C': [0.1, 1, 5, 10, 50],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'kernel': ['rbf'] # RBF is typically optimal for non-linear datasets like the Titanic
}

# 5-fold cross-validation keeping class ratios consistent
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid Search setup
grid_search = GridSearchCV(
    estimator=SVC(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Run Grid Search
grid_search.fit(X_train_scaled, y_train)

print("Best Parameters Found:", grid_search.best_params_)
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best Parameters Found: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Best Cross-Validation Accuracy: 0.8361


In [6]:
best_svc = grid_search.best_estimator_

# Check the model performance on our training split
y_train_pred = best_svc.predict(X_train_scaled)
print("\nTraining Set Classification Report:")
print(classification_report(y_train, y_train_pred))


Training Set Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.90      0.87       549
           1       0.82      0.74      0.78       342

    accuracy                           0.84       891
   macro avg       0.84      0.82      0.83       891
weighted avg       0.84      0.84      0.84       891



In [7]:
# Predict on the unseen test set
test_predictions = best_svc.predict(X_test_scaled)

# Create submission DataFrame
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': test_predictions
})

# Save the submission file
submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' has been created successfully!")

Submission file 'submission.csv' has been created successfully!
